# SHAP Chunk Generator

This notebook only loads the EMBER dataset/model, runs `model_utils.explain_model(...)` on deterministic row chunks, saves each chunk as a pickle, and optionally merges completed chunks into one full SHAP pickle.

It is intended for the LightGBM EMBER setup where SHAP is computed by `Booster.predict(..., pred_contrib=True)`. Chunking rows does not change LightGBM per-row SHAP values when chunks are merged back in the original row order.

Run one `CHUNK_INDEX` at a time. After all chunks exist, set `COMPUTE_CURRENT_CHUNK = False` and `MERGE_SHAP_CHUNKS = True` in the config cell, then run the merge cell.

In [36]:
from pathlib import Path
import json
import os
import random
import sys
import time


def find_repo_root(start=None):
    start = Path(start or Path.cwd()).resolve()
    for candidate in (start, *start.parents):
        if (candidate / 'backdoor_attack.py').exists() and (candidate / 'mw_backdoor').is_dir():
            return candidate
    raise RuntimeError('Could not locate repository root containing backdoor_attack.py and mw_backdoor/')


REPO_ROOT = find_repo_root()
os.chdir(REPO_ROOT)
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

os.environ.setdefault('MPLCONFIGDIR', str(REPO_ROOT / 'build' / 'matplotlib'))
os.makedirs(os.environ['MPLCONFIGDIR'], exist_ok=True)

import ember
import numpy as np
import pandas as pd
import tensorflow as tf

from mw_backdoor import common_utils
from mw_backdoor import constants
from mw_backdoor import data_utils
from mw_backdoor import model_utils

CONFIG_DIR = REPO_ROOT / 'configs'
DEFAULT_CONFIG_PATH = CONFIG_DIR / 'lightgbm_fig4_test.json'
if not DEFAULT_CONFIG_PATH.exists():
    DEFAULT_CONFIG_PATH = CONFIG_DIR / 'lightgbm_fig2.json'

EMBER_DATA_DIR = REPO_ROOT / 'ember2018'
SAVED_FILES_DIR = REPO_ROOT / 'saved_files'
EMBER_FEATURE_VERSION = 2
LIGHTGBM_MODEL_FILE = 'ember_model_2018.txt'
LIGHTGBM_MODEL_DIR = EMBER_DATA_DIR
SHAP_CACHE_DIR = EMBER_DATA_DIR
SEED = 42

constants.EMBER_DATA_DIR = str(EMBER_DATA_DIR)
constants.SAVE_FILES_DIR = str(SAVED_FILES_DIR)
constants.SAVE_MODEL_DIR = str(SAVED_FILES_DIR)
constants.NUM_EMBER_FEATURES = ember.PEFeatureExtractor(
    EMBER_FEATURE_VERSION,
    print_feature_warning=False,
).dim
constants.num_features['ember'] = constants.NUM_EMBER_FEATURES


def load_attack_dataset(dataset='ember', selected=False):
    if dataset != 'ember' or EMBER_FEATURE_VERSION == 1:
        return data_utils.load_dataset(dataset=dataset, selected=selected)

    try:
        x_train, y_train, x_test, y_test = ember.read_vectorized_features(
            constants.EMBER_DATA_DIR,
            feature_version=EMBER_FEATURE_VERSION,
        )
    except Exception:
        ember.create_vectorized_features(
            constants.EMBER_DATA_DIR,
            feature_version=EMBER_FEATURE_VERSION,
        )
        x_train, y_train, x_test, y_test = ember.read_vectorized_features(
            constants.EMBER_DATA_DIR,
            feature_version=EMBER_FEATURE_VERSION,
        )

    x_train = x_train.astype(dtype='float64')
    x_test = x_test.astype(dtype='float64')
    x_train = x_train[y_train != -1]
    y_train = y_train[y_train != -1]
    x_test = x_test[y_test != -1]
    y_test = y_test[y_test != -1]
    return x_train, y_train, x_test, y_test


def load_attack_model(model_id, data_id, save_path=None, file_name=None):
    if data_id == 'ember' and model_id == 'lightgbm' and LIGHTGBM_MODEL_FILE:
        return model_utils.load_model(
            model_id=model_id,
            data_id=data_id,
            save_path=str(LIGHTGBM_MODEL_DIR),
            file_name=LIGHTGBM_MODEL_FILE,
        )

    return model_utils.load_model(
        model_id=model_id,
        data_id=data_id,
        save_path=save_path or constants.SAVE_MODEL_DIR,
        file_name=file_name or data_id + '_' + model_id,
    )


print('Repository root:', REPO_ROOT)
print('Config path:', DEFAULT_CONFIG_PATH)
print('EMBER data dir:', constants.EMBER_DATA_DIR)
print('EMBER feature version:', EMBER_FEATURE_VERSION)
print('EMBER feature count:', constants.NUM_EMBER_FEATURES)


Repository root: /Users/falcon/Machine Learning/Severi Data Poisoning Attack
Config path: /Users/falcon/Machine Learning/Severi Data Poisoning Attack/configs/lightgbm_fig4_test.json
EMBER data dir: /Users/falcon/Machine Learning/Severi Data Poisoning Attack/ember2018
EMBER feature version: 2
EMBER feature count: 2381


In [37]:
cfg = common_utils.read_config(str(DEFAULT_CONFIG_PATH), atk_def=True)
cfg['seed'] = SEED

# This chunk notebook is for the current LightGBM/EMBER setup.
model_id = cfg['model']
dataset = cfg['dataset']
if dataset != 'ember' or model_id != 'lightgbm':
    raise NotImplementedError('This notebook currently supports only LightGBM on EMBER.')

# Use train for the logically clean attack setup used by the main notebook.
SHAP_DATA_SPLIT = 'train'
SHAP_PERCENT = 1.0

# For 5% chunks, use 20 chunks. Run CHUNK_INDEX 0, then 1, ... through 19.
SHAP_NUM_CHUNKS = 20
SHAP_CHUNK_INDEX = 19

# Set this False when you only want to merge already-created chunks.
COMPUTE_CURRENT_CHUNK = False

# Set this True only after all chunk pickle files exist.
MERGE_SHAP_CHUNKS = True

# Set True if you intentionally want to recompute an existing chunk file.
OVERWRITE_CHUNK = False

print('Config:', cfg)
print('SHAP_DATA_SPLIT:', SHAP_DATA_SPLIT)
print('SHAP_PERCENT:', SHAP_PERCENT)
print('SHAP_NUM_CHUNKS:', SHAP_NUM_CHUNKS)
print('SHAP_CHUNK_INDEX:', SHAP_CHUNK_INDEX)


Config: {'model': 'lightgbm', 'poison_size': [3000], 'watermark_size': [17], 'target_features': 'feasible', 'feature_selection': ['shap_largest_abs'], 'value_selection': ['argmin_Nv_sum_abs_shap'], 'iterations': 5, 'dataset': 'ember', 'k_perc': 1.0, 'k_data': 'train', 'seed': 42}
SHAP_DATA_SPLIT: train
SHAP_PERCENT: 1.0
SHAP_NUM_CHUNKS: 20
SHAP_CHUNK_INDEX: 19


In [38]:
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

x_train, y_train, x_test, y_test = load_attack_dataset(dataset=dataset, selected=True)
if SHAP_DATA_SPLIT == 'train':
    x_shap, y_shap = x_train, y_train
elif SHAP_DATA_SPLIT == 'test':
    x_shap, y_shap = x_test, y_test
else:
    raise ValueError('Unsupported SHAP_DATA_SPLIT: {}'.format(SHAP_DATA_SPLIT))

if SHAP_PERCENT != 1.0:
    n_rows = int(round(x_shap.shape[0] * SHAP_PERCENT))
    x_shap = x_shap[:n_rows]
    y_shap = y_shap[:n_rows]

original_model = load_attack_model(
    model_id=model_id,
    data_id=dataset,
    save_path=constants.SAVE_MODEL_DIR,
    file_name=dataset + '_' + model_id,
)

base_cache_key = '{}_{}_{}_{}_v{}'.format(
    dataset,
    model_id,
    SHAP_DATA_SPLIT,
    str(SHAP_PERCENT).replace('.', 'p'),
    EMBER_FEATURE_VERSION,
)
merged_shap_path = SHAP_CACHE_DIR / 'shap_values_{}.pkl'.format(base_cache_key)

print('Train:', x_train.shape, y_train.shape)
print('Test:', x_test.shape, y_test.shape)
print('SHAP source:', x_shap.shape, y_shap.shape)
print('Merged SHAP path:', merged_shap_path)


[LightGBM] [Warning] Ignoring unrecognized parameter 'max_conflict_rate' found in model string.
[LightGBM] [Warning] Ignoring unrecognized parameter 'sparse_threshold' found in model string.
[LightGBM] [Warning] Ignoring unrecognized parameter 'enable_load_from_binary_file' found in model string.
[LightGBM] [Warning] Ignoring unrecognized parameter 'max_position' found in model string.
Train: (600000, 2381) (600000,)
Test: (200000, 2381) (200000,)
SHAP source: (600000, 2381) (600000,)
Merged SHAP path: /Users/falcon/Machine Learning/Severi Data Poisoning Attack/ember2018/shap_values_ember_lightgbm_train_1p0_v2.pkl


In [39]:
def chunk_bounds(n_rows, num_chunks, chunk_index):
    if not 0 <= chunk_index < num_chunks:
        raise ValueError('chunk_index must be in [0, num_chunks)')
    start = n_rows * chunk_index // num_chunks
    end = n_rows * (chunk_index + 1) // num_chunks
    return start, end


chunk_start, chunk_end = chunk_bounds(x_shap.shape[0], SHAP_NUM_CHUNKS, SHAP_CHUNK_INDEX)
chunk_path = SHAP_CACHE_DIR / 'shap_values_{}_chunk_{:03d}_of_{:03d}.pkl'.format(
    base_cache_key,
    SHAP_CHUNK_INDEX,
    SHAP_NUM_CHUNKS,
)
metadata_path = chunk_path.with_suffix('.json')

print('Chunk rows: [{}:{})'.format(chunk_start, chunk_end))
print('Chunk path:', chunk_path)

if COMPUTE_CURRENT_CHUNK:
    if chunk_path.exists() and not OVERWRITE_CHUNK:
        print('Chunk already exists and OVERWRITE_CHUNK is False. Skipping compute.')
    else:
        x_chunk = x_shap[chunk_start:chunk_end]
        print('Computing SHAP for chunk shape:', x_chunk.shape)
        start_time = time.time()
        shap_chunk_df = model_utils.explain_model(
            data_id=dataset,
            model_id=model_id,
            model=original_model,
            x_exp=x_chunk,
            x_back=x_chunk,
            perc=1.0,
            n_samples=100,
            load=False,
            save=False,
        )
        print('Computing chunk SHAP took {:.2f} seconds'.format(time.time() - start_time))
        assert shap_chunk_df.shape == x_chunk.shape, (
            'Chunk SHAP shape {} does not match chunk data shape {}'.format(shap_chunk_df.shape, x_chunk.shape)
        )
        shap_chunk_df.index = range(chunk_start, chunk_end)
        start_time = time.time()
        shap_chunk_df.to_pickle(chunk_path)
        metadata = {
            'base_cache_key': base_cache_key,
            'chunk_index': SHAP_CHUNK_INDEX,
            'num_chunks': SHAP_NUM_CHUNKS,
            'row_start': chunk_start,
            'row_end': chunk_end,
            'shape': list(shap_chunk_df.shape),
            'dataset': dataset,
            'model_id': model_id,
            'shap_data_split': SHAP_DATA_SPLIT,
            'shap_percent': SHAP_PERCENT,
            'ember_feature_version': EMBER_FEATURE_VERSION,
        }
        metadata_path.write_text(json.dumps(metadata, indent=2) + '\n')
        print('Saving chunk pickle took {:.2f} seconds'.format(time.time() - start_time))
        print('Saved:', chunk_path)
        print('Saved metadata:', metadata_path)
else:
    print('COMPUTE_CURRENT_CHUNK is False. No chunk was computed.')


Chunk rows: [570000:600000)
Chunk path: /Users/falcon/Machine Learning/Severi Data Poisoning Attack/ember2018/shap_values_ember_lightgbm_train_1p0_v2_chunk_019_of_020.pkl
COMPUTE_CURRENT_CHUNK is False. No chunk was computed.


In [40]:
chunk_paths = [
    SHAP_CACHE_DIR / 'shap_values_{}_chunk_{:03d}_of_{:03d}.pkl'.format(
        base_cache_key,
        chunk_index,
        SHAP_NUM_CHUNKS,
    )
    for chunk_index in range(SHAP_NUM_CHUNKS)
]
missing_chunks = [path for path in chunk_paths if not path.exists()]

print('Expected chunks:', len(chunk_paths))
print('Missing chunks:', len(missing_chunks))
if missing_chunks:
    print('First missing chunk:', missing_chunks[0])

if MERGE_SHAP_CHUNKS:
    if missing_chunks:
        raise FileNotFoundError('Cannot merge because {} chunk(s) are missing.'.format(len(missing_chunks)))

    start_time = time.time()
    print('Loading and merging chunk pickles...')
    shap_values_df = pd.concat((pd.read_pickle(path) for path in chunk_paths), axis=0)
    shap_values_df = shap_values_df.sort_index().reset_index(drop=True)
    assert shap_values_df.shape == x_shap.shape, (
        'Merged SHAP shape {} does not match source data shape {}'.format(shap_values_df.shape, x_shap.shape)
    )
    shap_values_df.to_pickle(merged_shap_path)
    print('Merged SHAP pickle took {:.2f} seconds'.format(time.time() - start_time))
    print('Merged SHAP shape:', shap_values_df.shape)
    print('Saved merged pickle:', merged_shap_path)
else:
    print('MERGE_SHAP_CHUNKS is False. No merge was performed.')


Expected chunks: 20
Missing chunks: 0
Loading and merging chunk pickles...
Merged SHAP pickle took 331.63 seconds
Merged SHAP shape: (600000, 2381)
Saved merged pickle: /Users/falcon/Machine Learning/Severi Data Poisoning Attack/ember2018/shap_values_ember_lightgbm_train_1p0_v2.pkl
